# Lab 3 — Interrogate Your Photos
**Session 3 · Multimodal · TCE** · First: **File → Save a copy in Drive**.

You need 2–3 photos: anything on your phone — a receipt, your handwritten notes, the canteen menu board. Transfer to laptop (email yourself / Drive / USB cable).

In [ ]:
# Cell 1 — setup
%pip install -q -U google-genai pillow
from getpass import getpass
from google import genai
from google.genai import types
from PIL import Image
import time

client = genai.Client(api_key=getpass("Gemini API key: "))
MODEL = "gemini-flash-latest"  # the free tier's current Flash (July 2026 → Gemini 3.5 Flash). 503 'high demand'? swap to "gemini-flash-lite-latest".

def _shrink(x, cap=1024):
    """Downscale big images before sending — saves free-tier quota and time.
    A 4000px phone photo and a 1024px one give the model the same answer."""
    if isinstance(x, Image.Image) and max(x.size) > cap:
        r = cap / max(x.size)
        return x.resize((int(x.width*r), int(x.height*r)))
    return x

def ask(contents, temperature=None):
    """contents can be a string, or a list mixing PIL Images and strings."""
    if isinstance(contents, list):
        contents = [_shrink(c) for c in contents]
    config = types.GenerateContentConfig(temperature=temperature) if temperature is not None else None
    for attempt in range(4):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config).text
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited..."); time.sleep(20*(attempt+1))
            else: raise
print("ready ✓")

## Part A — Photo Q&A (escalating difficulty)

Upload a photo: **folder icon (left sidebar) → upload**. Then run the ladder.

In [ ]:
# Cell 2 — the interrogation ladder
img = Image.open("your_photo.jpg")   # ← your filename
display(img.resize((min(400, img.width), int(img.height * min(400, img.width) / img.width))))

questions = [
    "Describe this image in 2 sentences.",
    "Read ALL text visible in this image, exactly as written.",
    "How many distinct objects/people are in this image? Count carefully.",
    "What can you infer about where and when this was taken?",
    "What is the most surprising detail in this image?",
]
for q in questions:
    print("=" * 60, "\nQ:", q)
    print(ask([img, q]), "\n")

### Grade it (edit this cell)
Which answers were right? Where did it wobble — counting? small text? inference?

### ✓ Checkpoint 1 — ladder run + your 2-line grading.

---
## Part B — Receipt / document → JSON

Session 2's format control, now with eyes. Strict schema, `ONLY`, grounding line.

In [ ]:
# Cell 3 — structured extraction
doc = Image.open("receipt.jpg")   # ← receipt / bill / marksheet / form

schema_prompt = """Extract data from this image.
Reply ONLY with JSON in exactly this schema:
{"vendor": str, "date": str, "items": [{"name": str, "price": float}], "total": float}
If any field is unreadable, use null — do NOT guess."""

print(ask([doc, schema_prompt], temperature=0.0))

In [ ]:
# Cell 4 — prove it parses (the real test)
import json as pyjson
raw = ask([doc, schema_prompt], temperature=0.0)
# strip accidental code fences if present
raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
data = pyjson.loads(raw)
print("PARSED ✓  total =", data["total"])
# If this cell crashes, your prompt isn't strict enough. Tighten and re-run — that's the lesson.

### ✓ Checkpoint 2 — `json.loads` succeeds on your document.

---
## Part B2 — the production way: guaranteed JSON

Begging for JSON in the prompt works — you just proved it. Production code doesn't beg: it passes a **schema** with the request. The API then **cannot** return anything else — no code fences, no "Certainly!", nothing to strip — and you can delete the format instructions from your prompt entirely. The prompt says *what* to extract; the schema locks the *shape*.

In [ ]:
# Cell 4b — response_schema: the API cannot answer in anything but your JSON
schema = {
    "type": "OBJECT",
    "properties": {
        "vendor": {"type": "STRING"},
        "date":   {"type": "STRING"},
        "total":  {"type": "NUMBER"},
        "items":  {"type": "ARRAY", "items": {
            "type": "OBJECT",
            "properties": {"name": {"type": "STRING"}, "price": {"type": "NUMBER"}},
            "required": ["name", "price"],
        }},
    },
    "required": ["vendor", "total"],
}

resp = client.models.generate_content(
    model=MODEL,
    contents=[_shrink(doc), "Extract the receipt."],   # ← no format instructions at all
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=schema,
    ),
)
data = pyjson.loads(resp.text)   # no fence-stripping, no crash — ever
print("PARSED ✓  vendor =", data["vendor"], "· total =", data["total"])

---
## Part C — Handwriting test

Photograph a page of YOUR handwritten notes → transcribe → grade yourself: roughly what % correct? Tamil/Tanglish notes = bonus experiment.

In [ ]:
# Cell 5 — handwriting
notes = Image.open("my_notes.jpg")
print(ask([notes, "Transcribe this handwritten page exactly. Mark unclear words as [?]."]))

## Part D — Break it

Find one image where the model **confidently invents a detail**: a blurred price it "reads" anyway, objects it miscounts, text it paraphrases instead of quoting.

### ✓ Checkpoint 3 — show me the invention.

---
## Stretch goals

In [ ]:
# Stretch 1 — does grounding stop the invention?
# Re-run your Part D image WITH the grounding line vs WITHOUT:
loose  = "What is the total on this receipt?"
strict = "What is the total on this receipt? If it is not clearly readable, reply exactly: UNREADABLE."
img_d = Image.open("your_partD_image.jpg")
print("loose :", ask([img_d, loose]))
print("strict:", ask([img_d, strict]))

In [ ]:
# Stretch 2 — vision eval (your S2 harness grows eyes)
vision_tests = [
    {"img": "receipt.jpg", "q": "What is the total?", "expected": "342"},
    # add 4 more: photo, question, expected key fact
]
import re
def norm(s): return re.sub(r"[^a-z0-9 ]", "", s.lower())
hits = 0
for t in vision_tests:
    ans = ask([Image.open(t["img"]), t["q"]], temperature=0.0)
    ok = norm(t["expected"]) in norm(ans); hits += ok
    print("✓" if ok else "✗", t["q"], "→", ans[:60])
print(f"vision score: {hits}/{len(vision_tests)}")

In [ ]:
# Stretch 3 — audio: record a short voice note on your phone, upload it
audio = client.files.upload(file="voicenote.m4a")
print(ask([audio, "Transcribe this audio, then summarize it in one line."]))

## Day 1 complete

**Tonight (5 min, mandatory):** put 2–3 real documents (lecture notes, textbook chapter PDF) on your laptop/Drive. Tomorrow: **chat with your notes** — the pattern behind most real AI products.

Sleep well. Day 2 is the good stuff.